# 📬 Notebook 2: Queue Basics

Use Redis as a job queue to decouple request acceptance from processing.

## Learning Objectives

By the end of this notebook, you'll understand:
- How job queues work
- Submitting jobs to Redis
- Tracking job status in PostgreSQL
- The complete submission flow

In [ ]:
import redis
import psycopg2
import json
import uuid
from datetime import datetime
from typing import Optional

r = redis.Redis(host='localhost', port=6379, decode_responses=True)

conn = psycopg2.connect(
    host="localhost", port=5432,
    database="taskqueue", user="postgres", password="postgres"
)
conn.autocommit = True

print("✅ Connected to Redis and PostgreSQL!")
print("📊 Open RedisInsight: http://localhost:5540")
print("📊 Open Adminer: http://localhost:8080")

## 🔄 Reset State

Run this cell to clear all jobs, logs, and queue data. This ensures a clean slate for demos.

In [ ]:
def reset_all():
    r.flushall()
    cursor = conn.cursor()
    cursor.execute("DELETE FROM job_logs")
    cursor.execute("DELETE FROM dead_letter_queue")
    cursor.execute("DELETE FROM jobs")
    cursor.close()
    print("🔄 Reset complete: Redis flushed, jobs/logs/dlq cleared")

reset_all()

## 📬 Queue Architecture

In [ ]:
print("📬 How Job Queues Work")
print("=" * 60)
print("""
A job queue has two main operations:

PUSH (Producer/API Server):
─────────────────────────────────────────────────────────────
1. Validate request
2. Create job record in database (status: pending)
3. Push job_id to Redis queue
4. Return job_id to client immediately

                    PUSH
    [API Server] ─────────> [Redis Queue]
                            │job_1│job_2│job_3│

POP (Consumer/Worker):
─────────────────────────────────────────────────────────────
1. Pop job_id from queue (blocking)
2. Fetch job details from database
3. Update status to 'processing'
4. Do the actual work
5. Update status to 'completed' or 'failed'

                             POP
    [Redis Queue] ─────────> [Worker]
    │job_2│job_3│           Processing job_1...

WHY REDIS?
• Fast (in-memory)
• Persistent (AOF/RDB)
• Blocking pop (BRPOP)
• Atomic operations
""")

## 📝 Job Submission Service

In [ ]:
QUEUE_NAME = "jobs:pending"

class JobService:
    def __init__(self, redis_client, db_conn):
        self.redis = redis_client
        self.conn = db_conn
    
    def submit_job(self, job_type: str, payload: dict, priority: int = 0) -> dict:
        job_id = str(uuid.uuid4())
        
        cursor = self.conn.cursor()
        cursor.execute("""
            INSERT INTO jobs (id, job_type, payload, priority, status)
            VALUES (%s, %s, %s, %s, 'pending')
            RETURNING id, created_at
        """, (job_id, job_type, json.dumps(payload), priority))
        row = cursor.fetchone()
        cursor.close()
        
        self.redis.lpush(QUEUE_NAME, job_id)
        
        return {
            'job_id': job_id,
            'status': 'pending',
            'created_at': str(row[1])
        }
    
    def get_job_status(self, job_id: str) -> Optional[dict]:
        cursor = self.conn.cursor()
        cursor.execute("""
            SELECT id, job_type, status, result, error_message, 
                   attempts, created_at, completed_at
            FROM jobs WHERE id = %s
        """, (job_id,))
        row = cursor.fetchone()
        cursor.close()
        
        if not row:
            return None
        
        return {
            'job_id': str(row[0]),
            'job_type': row[1],
            'status': row[2],
            'result': row[3],
            'error': row[4],
            'attempts': row[5],
            'created_at': str(row[6]),
            'completed_at': str(row[7]) if row[7] else None
        }
    
    def get_queue_stats(self) -> dict:
        queue_length = self.redis.llen(QUEUE_NAME)
        
        cursor = self.conn.cursor()
        cursor.execute("""
            SELECT status, COUNT(*) FROM jobs GROUP BY status
        """)
        status_counts = dict(cursor.fetchall())
        cursor.close()
        
        return {
            'queue_length': queue_length,
            'status_counts': status_counts
        }

job_service = JobService(r, conn)
print("✅ JobService ready!")

In [ ]:
print("📝 Submitting Jobs Demo")
print("=" * 60)

print("\n1️⃣ Submit a PDF generation job...")
job1 = job_service.submit_job(
    job_type="generate_pdf",
    payload={"user_id": "user_123", "report_type": "annual"}
)
print(f"   Job ID: {job1['job_id']}")
print(f"   Status: {job1['status']}")

print("\n2️⃣ Submit a video transcoding job...")
job2 = job_service.submit_job(
    job_type="transcode_video",
    payload={"video_id": "vid_456", "resolutions": ["1080p", "720p", "480p"]}
)
print(f"   Job ID: {job2['job_id']}")
print(f"   Status: {job2['status']}")

print("\n3️⃣ Submit a bulk email job...")
job3 = job_service.submit_job(
    job_type="send_bulk_email",
    payload={"template": "newsletter", "recipient_count": 50000}
)
print(f"   Job ID: {job3['job_id']}")
print(f"   Status: {job3['status']}")

print("\n📊 Queue Stats:")
stats = job_service.get_queue_stats()
print(f"   Jobs in queue: {stats['queue_length']}")
print(f"   Status counts: {stats['status_counts']}")

## 🔍 Check Job Status

In [ ]:
print("🔍 Checking Job Status")
print("=" * 60)

status = job_service.get_job_status(job1['job_id'])

print(f"\n📋 Job Details:")
print(f"   ID: {status['job_id']}")
print(f"   Type: {status['job_type']}")
print(f"   Status: {status['status']}")
print(f"   Attempts: {status['attempts']}")
print(f"   Created: {status['created_at']}")

print("\n📊 This is what your API would return:")
print(f"   GET /jobs/{job1['job_id'][:8]}...")
print(f"   Response: {{'status': '{status['status']}', 'progress': null}}")

## 📋 View Queue in Redis

In [ ]:
print("📋 Redis Queue Contents")
print("=" * 60)

queue_items = r.lrange(QUEUE_NAME, 0, -1)

print(f"\n🔑 Queue: {QUEUE_NAME}")
print(f"   Length: {len(queue_items)}")
print("\n   Jobs in queue (FIFO order):")
for i, job_id in enumerate(reversed(queue_items), 1):
    status = job_service.get_job_status(job_id)
    print(f"   {i}. {job_id[:8]}... ({status['job_type']})")

print("\n💡 Open RedisInsight to see the queue visually!")
print("   http://localhost:5540")

## 🔄 The Complete Flow

In [ ]:
print("🔄 Complete Job Submission Flow")
print("=" * 60)
print("""
What happens when user clicks "Generate Report":

┌─────────────────────────────────────────────────────────────┐
│ 1. API SERVER receives POST /generate-report               │
│    • Validates request (user auth, parameters)             │
│    • Takes: ~10ms                                          │
└─────────────────────────────────────────────────────────────┘
                           │
                           ▼
┌─────────────────────────────────────────────────────────────┐
│ 2. CREATE JOB RECORD in PostgreSQL                         │
│    • INSERT INTO jobs (id, type, payload, status='pending')│
│    • Takes: ~5ms                                           │
└─────────────────────────────────────────────────────────────┘
                           │
                           ▼
┌─────────────────────────────────────────────────────────────┐
│ 3. PUSH TO QUEUE in Redis                                  │
│    • LPUSH jobs:pending job_id                             │
│    • Takes: ~1ms                                           │
└─────────────────────────────────────────────────────────────┘
                           │
                           ▼
┌─────────────────────────────────────────────────────────────┐
│ 4. RETURN RESPONSE to client                               │
│    • {"job_id": "abc-123", "status": "pending"}            │
│    • Total time: ~20ms (not 45 seconds!)                   │
└─────────────────────────────────────────────────────────────┘

✅ User gets immediate feedback!
✅ Server is free to handle other requests!
✅ Job waits in queue for worker!
""")

## 🧪 Quick Quiz

1. **Why store jobs in PostgreSQL if they're also in Redis?**

2. **Why use LPUSH/RPOP instead of just a list?**

3. **What if Redis crashes after DB insert but before queue push?**

In [ ]:
print("📝 Quiz Answers")
print("=" * 50)
print()
print("1. Why both PostgreSQL and Redis:")
print("   - PostgreSQL: Durable storage, complex queries")
print("   - Redis: Fast queue operations, blocking pop")
print("   - Job data in DB, only ID in queue")
print()
print("2. LPUSH/RPOP for FIFO:")
print("   - LPUSH adds to left (head)")
print("   - RPOP removes from right (tail)")
print("   - First in, first out order")
print()
print("3. Crash between DB and Redis:")
print("   - Job exists in DB with 'pending' status")
print("   - Reconciliation job finds orphans")
print("   - Re-queues jobs stuck in 'pending'")

## 📚 Summary

### Key Takeaways

1. **Queue decouples** - Accept fast, process later
2. **Redis for queue** - Fast, persistent, blocking pop
3. **PostgreSQL for state** - Durable, queryable
4. **Only IDs in queue** - Data stays in database
5. **Return immediately** - 20ms not 45 seconds

### Next Up

In **Notebook 3**, we'll implement workers:
- Polling the queue
- Processing jobs
- Updating status